<a href="https://colab.research.google.com/github/tanu-1906/data_science_lab_1/blob/main/Pythonprograms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BFS & DFS

### Breadth First Search (BFS)

Breadth-First Search (BFS) is an algorithm for traversing or searching tree or graph data structures. It starts at the tree root (or some arbitrary node of a graph, sometimes referred to as a 'search key') and explores all of the neighbor nodes at the present depth prior to moving on to the nodes at the next depth level.

BFS uses a queue data structure to keep track of the nodes to visit.

In [1]:
from collections import deque

def bfs(graph, start_node):
    visited = set()  # To keep track of visited nodes
    queue = deque([start_node])  # Initialize queue with the starting node
    visited.add(start_node)

    bfs_path = []

    while queue:
        current_node = queue.popleft()  # Dequeue a node
        bfs_path.append(current_node)

        # Visit all unvisited neighbors
        for neighbor in graph.get(current_node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return bfs_path

# Example Graph
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F'],
    'F': ['C', 'E']
}

print("BFS starting from node A:", bfs(graph, 'A'))

BFS starting from node A: ['A', 'B', 'C', 'D', 'E', 'F']


### Depth First Search (DFS)

Depth-First Search (DFS) is an algorithm for traversing or searching tree or graph data structures. The algorithm starts at the root node (selecting some arbitrary node as the root node in the case of a graph) and explores as far as possible along each branch before backtracking.

DFS typically uses a stack data structure (or recursion, which uses the call stack) to keep track of the nodes to visit.

In [2]:
def dfs(graph, start_node):
    visited = set()  # To keep track of visited nodes
    stack = [start_node]  # Initialize stack with the starting node
    dfs_path = []

    while stack:
        current_node = stack.pop()  # Pop a node from the stack

        if current_node not in visited:
            visited.add(current_node)
            dfs_path.append(current_node)

            # Push unvisited neighbors onto the stack
            # Important: For consistent output, iterate in reverse or sort neighbors
            # so that smaller/earlier nodes are processed first when popped.
            for neighbor in sorted(graph.get(current_node, []), reverse=True):
                if neighbor not in visited:
                    stack.append(neighbor)
    return dfs_path

# Using the same example graph
print("DFS starting from node A:", dfs(graph, 'A'))

DFS starting from node A: ['A', 'B', 'D', 'E', 'F', 'C']


# A* Algorithm

## A* Algorithm

A* (pronounced "A-star") is a graph traversal and path search algorithm, which is often used in many fields of computer science due to its completeness, optimality, and optimal efficiency. It is an informed search algorithm, meaning it uses a heuristic function to guide its search.

### Key Concepts:

*   **Nodes**: Represent points in the search space.
*   **Cost (`g_cost`)**: The cost from the starting node to the current node.
*   **Heuristic (`h_cost`)**: An estimated cost from the current node to the goal node. It must be admissible (never overestimate the cost).
*   **Total Cost (`f_cost`)**: `g_cost + h_cost`. The algorithm prioritizes nodes with lower `f_cost`.
*   **Open List (Priority Queue)**: Contains nodes that have been discovered but not yet evaluated.
*   **Closed List**: Contains nodes that have already been evaluated.

In [3]:
import heapq # For priority queue

class Node:
    def __init__(self, state, parent=None, g_cost=0, h_cost=0):
        self.state = state
        self.parent = parent
        self.g_cost = g_cost  # Cost from start to this node
        self.h_cost = h_cost  # Heuristic cost from this node to goal
        self.f_cost = self.g_cost + self.h_cost # Total estimated cost

    # For comparison in priority queue (heapq prioritizes smaller f_cost)
    def __lt__(self, other):
        return self.f_cost < other.f_cost

def a_star_search(graph, start, goal, heuristic):
    open_list = [] # Priority queue (min-heap) of nodes to explore
    heapq.heappush(open_list, Node(start, g_cost=0, h_cost=heuristic[start]))

    closed_list = set() # Set of visited states

    g_costs = {start: 0} # Stores the g_cost of reaching each state
    parents = {start: None} # Stores the parent of each state in the optimal path

    while open_list:
        current_node = heapq.heappop(open_list)

        if current_node.state == goal:
            path = []
            while current_node:
                path.append(current_node.state)
                current_node = current_node.parent
            return path[::-1] # Reverse to get path from start to goal

        if current_node.state in closed_list:
            continue

        closed_list.add(current_node.state)

        for neighbor_state, edge_cost in graph.get(current_node.state, []):
            # Calculate tentative g_cost for neighbor
            tentative_g_cost = current_node.g_cost + edge_cost

            # If this path to neighbor is better than any previous one
            if neighbor_state not in g_costs or tentative_g_cost < g_costs[neighbor_state]:
                g_costs[neighbor_state] = tentative_g_cost
                parents[neighbor_state] = current_node

                # Create new node for neighbor and push to open list
                neighbor_h_cost = heuristic.get(neighbor_state, float('inf'))
                neighbor_node = Node(neighbor_state, current_node, tentative_g_cost, neighbor_h_cost)
                heapq.heappush(open_list, neighbor_node)

    return None # No path found

# Example Graph (Adjacency List with costs)
# Format: {node: [(neighbor, cost), ...]}
graph_astar = {
    'A': [('B', 1), ('C', 3)],
    'B': [('D', 3), ('E', 2)],
    'C': [('F', 5)],
    'D': [('G', 4)],
    'E': [('G', 2)],
    'F': [('G', 1)]
}

# Heuristic function (straight-line distance to goal 'G')
heuristic_g = {
    'A': 7,
    'B': 6,
    'C': 8,
    'D': 4,
    'E': 2,
    'F': 1,
    'G': 0 # Goal node has heuristic 0
}

start_node_astar = 'A'
goal_node_astar = 'G'

path = a_star_search(graph_astar, start_node_astar, goal_node_astar, heuristic_g)

if path:
    print(f"A* path from {start_node_astar} to {goal_node_astar}: {path}")
else:
    print(f"No path found from {start_node_astar} to {goal_node_astar}")

A* path from A to G: ['A', 'B', 'E', 'G']


# Minmax algorithm

# Minimax Algorithm

Minimax is a decision-making algorithm, typically used in artificial intelligence for two-player game theory. It is used to choose an optimal move for a player, assuming the opponent also plays optimally. The algorithm works by recursively exploring the game tree and assigning a value to each node (game state).

### Key Concepts:

*   **Game Tree**: A tree representation of all possible sequences of moves and states in a game.
*   **Terminal Nodes**: Nodes at the end of the game tree, representing final game states (e.g., win, loss, draw). Each terminal node has a utility value.
*   **Utility Value**: A numerical score assigned to terminal nodes, representing the desirability of that state for the maximizing player.
*   **Maximizing Player**: The player who tries to maximize the utility value.
*   **Minimizing Player**: The player who tries to minimize the utility value (which is equivalent to maximizing their own utility value if the game is zero-sum).
*   **Recursion**: The algorithm works by recursively evaluating nodes from the leaves up to the root.

In [5]:
def minimax(node, depth, maximizing_player, game_tree):
    # Base case: If node is a terminal node or depth limit is reached
    if depth == 0 or node not in game_tree:
        return node, game_tree.get(node, 0) # Return node and its value (0 if not terminal)

    if maximizing_player:
        max_eval = -float('inf')
        best_move = None
        for child in game_tree[node]:
            _, eval = minimax(child, depth - 1, False, game_tree)
            if eval > max_eval:
                max_eval = eval
                best_move = child
        return best_move, max_eval
    else:
        min_eval = float('inf')
        best_move = None
        for child in game_tree[node]:
            _, eval = minimax(child, depth - 1, True, game_tree)
            if eval < min_eval:
                min_eval = eval
                best_move = child
        return best_move, min_eval

# Example Game Tree:
# This tree represents a simple game where the maximizing player wants a higher score
# and the minimizing player wants a lower score. Terminal nodes have numeric values.
# Non-terminal nodes point to lists of their children.
game_tree = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F', 'G'],
    'D': 3,
    'E': 12,
    'F': 8,
    'G': 2
}

# Let's adjust the depth and the interpretation for the example
# We assume A is the root, and we want to find the best move from A
# The depth here means how many moves ahead we look.
# For the example, 'D', 'E', 'F', 'G' are terminal nodes.

# Example usage:
# Starting from root 'A', depth 2, maximizing player
# The structure of the game_tree above has values directly at terminal nodes.
# So, the recursive call needs to properly interpret depth.
# Let's redefine the tree to explicitly show children for all non-terminal nodes
# and have the values only at the leaves.

# A more typical game tree structure for minimax:
# Each node is a state, and children are possible next states.
# The values are only at the actual leaf nodes (game end states).

# Let's create a game tree where leaf nodes have scores directly
# This game has 2 levels of moves after the root, so depth = 2 from the root.
# The root is 'A', first move by MAX, second by MIN, then terminal states.
minimax_game_tree = {
    'A': ['B', 'C'],        # MAX's turn
    'B': ['D', 'E'],        # MIN's turn
    'C': ['F', 'G'],        # MIN's turn
    'D': 3,                 # Terminal value
    'E': 12,                # Terminal value
    'F': 8,                 # Terminal value
    'G': 2                  # Terminal value
}

# Max player starts from 'A', depth 2 means looking two moves ahead (A -> B/C -> D/E/F/G)
initial_state = 'A'
depth = 2 # The number of levels to look down the tree

best_move, optimal_value = minimax(initial_state, depth, True, minimax_game_tree)

print(f"For the maximizing player starting at '{initial_state}':")
print(f"Optimal value: {optimal_value}")
print(f"Best move: {best_move}")

For the maximizing player starting at 'A':
Optimal value: 3
Best move: B
